In [22]:
import joblib
import pandas as pd
from sklearn import set_config
from tempfile import TemporaryDirectory
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.utils.class_weight import compute_sample_weight
import warnings

In [23]:
target_column = "health_condition"

In [24]:
X_train = pd.read_csv("../data/intermediate/train_features_oheencoded.csv")
X_valid = pd.read_csv("../data/intermediate/valid_features_oheencoded.csv")
X_test = pd.read_csv("../data/intermediate/test_features_oheencoded.csv")

y_train = pd.read_csv("../data/intermediate/train_labels.csv")
y_valid = pd.read_csv("../data/intermediate/valid_labels.csv")

X_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 552070 entries, 0 to 552069
Data columns (total 24 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   sleep_duration                491266 non-null  float64
 1   heart_rate                    545802 non-null  float64
 2   bmi                           541053 non-null  float64
 3   calorie_expenditure           509705 non-null  float64
 4   step_count                    540909 non-null  float64
 5   exercise_duration             546550 non-null  float64
 6   water_intake                  517224 non-null  float64
 7   stress_level                  485727 non-null  float64
 8   sleep_quality                 505469 non-null  float64
 9   physical_activity_level       522744 non-null  float64
 10  smoking_alcohol               529184 non-null  float64
 11  calorie_expenditure_per_step  499392 non-null  float64
 12  step_speed                    535492 non-null  float64


In [25]:
X = pd.concat([X_train, X_valid], axis=0)
y = pd.concat([y_train, y_valid], axis=0)

In [26]:
df_submission = pd.read_csv("../data/sample_submission.csv")
df_submission.info()

<class 'pandas.DataFrame'>
RangeIndex: 295753 entries, 0 to 295752
Data columns (total 2 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   id                295753 non-null  int64
 1   health_condition  295753 non-null  str  
dtypes: int64(1), str(1)
memory usage: 4.5 MB


In [27]:
y_train

,health_condition
0,1
1,2
2,1
3,1
4,1
...,...
552065,1
552066,1
552067,1
552068,1


In [28]:
train_sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
valid_sample_weight = compute_sample_weight(class_weight="balanced", y=y_valid)
y_sample_weight = compute_sample_weight(class_weight="balanced", y=y)

model = RandomForestClassifier(n_estimators=100, criterion='gini', max_depth=8)
model.fit(X_train, y_train, sample_weight=train_sample_weight)

joblib.dump(model, '../models/random_forest_80.pkl')

c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


['../models/random_forest_80.pkl']

In [ ]:
from sklearn.metrics import balanced_accuracy_score

y_pred = model.predict(X_valid)

val_score = balanced_accuracy_score(y_valid, y_pred, sample_weight=valid_sample_weight)
print("Validation Balanced Accuracy:", val_score)

In [29]:
model = RandomForestClassifier(n_estimators=100, criterion='gini', max_depth=8)
model.fit(X, y, sample_weight=y_sample_weight)

joblib.dump(model, '../models/random_forest_100.pkl')

c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


['../models/random_forest_100.pkl']

In [30]:
y_pred = model.predict(X_test)

df_submission[target_column] = y_pred
df_submission[target_column] = df_submission[target_column].replace({0:'unhealthy', 1:'at-risk', 2: 'fit'})

df_submission.to_csv('../results/random_forest_baseline.csv', index=False)
df_submission

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
...,...,...
295748,985836,fit
295749,985837,at-risk
295750,985838,unhealthy
295751,985839,at-risk
